# Task 1, Exercise 3
<p align="center">
  <img src="../src/resources/img/a1_u3_img.PNG" alt="Schwingungssystem", height="300">
</p>

A mass $m_0$​, which is supported by a spring-damper element (spring stiffness $c$, damping constant $d$) and guided laterally, is excited into oscillations by a rotating, eccentrically mounted mass point $m_{u}$​ (eccentricity $\varepsilon$).


1. Draw a free-body diagram of the system and indicate all forces
2. Derive the equation of motion for the system. Use the general form: $ \ddot{z} + 2\delta \dot{z} + \omega_0^2 z = b_0 u + b_1 \dot{u} + b_2 \ddot{u}$
3. Determine and sketch the amplitude response $V(\Omega)$
4. What should the damping ratio $D$ be so that the amplitude ratio $\hat{z} / \varepsilon = 2$ is obtained at resonance?
5. Determine the amplitude response from the unbalance excitation to the ground force. At what point does active vibration isolation occur?

Given: $c$, $d$, $\varepsilon$, $m_0$, $m_u$

Related topics: 
- [Rotating Imbalance](https://hub.gesis.mybinder.org/user/strukturdynamik-nische-mechanik-st12hovp/lab/tree/Jupyter%20Notebooks/Rotating_Imbalance.ipynb)


In [1]:
import numpy as np

In [ ]:
%matplotlib widget
%load_ext autoreload
%autoreload 2
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "src")))

import warnings
warnings.filterwarnings("ignore", message="findfont: Font family")

In [3]:
z0 = 0.0 # initial defl 
z0_dot = 0  # init velocity
m_0 = 100  
m_u = 10
e = 1
c = 5000
d = 10
m = m_0 + m_u
omega_0 = np.sqrt(c/m)
Omega = omega_0
alpha = Omega/10
b2 = -m_u/m
delta = d/(2*m)


In [ ]:
from scipy.integrate import odeint

from src.project.calculations.validate_solution import validate_solution

correct = False


class StudentSolver:
    def __init__(self) -> None:
        return None

    def state_space_steady(self, x, t, m_u, m, d, c, e):
        delta = d/(2*m)
        omega_0 = np.sqrt(c/m)
        Omega = omega_0
        b2 = -m_u/m
        [z, z_p] = x
        # write your code here
        x_p = None
        return x_p

    def state_space_accelerated(self, x, t, m_u, m, d, c, e, alpha):
        delta = d/(2*m)
        omega_0 = np.sqrt(c/m)
        b2 = -m_u/m
        [z, z_p] = x
        x_p = [z_p, -2*delta*z_p - omega_0**2*z - b2*e*(alpha*t)**2*np.sin(0.5*alpha*t**2)]
        return x_p

    def integrate(self, func, start_deflection, start_velocity, t, *args):

        x0 = (start_deflection, start_velocity)
        return odeint(func=func, y0=x0, t=t, args=args)[:, 0]


from src.project.calculations.int_solver_a1_u3 import IntSolverAufgabe1Uebung3 as IntSolverCorrect

np.set_printoptions(precision=4, suppress=True, linewidth=120)
student_solver = StudentSolver()
correct_solver = IntSolverCorrect()
t = np.linspace(0, 20, 500)


# settled
settled_student = student_solver.integrate(
                                student_solver.state_space_steady, 
                                z0,
                                z0_dot,
                                t,
                                m_u,
                                m,
                                d, 
                                c,
                                e)

settled_correct = correct_solver.integrate(
                                student_solver.state_space_steady, 
                                z0,
                                z0_dot,
                                t,
                                m_u,
                                m,
                                d, 
                                c,
                                e)

# accelerated
accelerated_student = student_solver.integrate(
                                student_solver.state_space_accelerated, 
                                z0,
                                z0_dot,
                                t,
                                m_u,
                                m,
                                d, 
                                c,
                                e,
                                alpha)

accelerated_correct = correct_solver.integrate(
                                student_solver.state_space_accelerated, 
                                z0,
                                z0_dot,
                                t,
                                m_u,
                                m,
                                d, 
                                c,
                                e,
                                alpha)


# test results
print("Settled: ")
correct_settled = validate_solution(settled_correct, settled_student, relative_threshold=0.05)
print()
print("Accelerated: ")
correct_acc = validate_solution(accelerated_correct, accelerated_student, relative_threshold=0.05)

# solution is correct if both settled and accelerated are correct
correct = correct_settled and correct_acc


In [ ]:
%matplotlib widget
%load_ext autoreload
%autoreload 2

from src.project.gui.gui_a1_u3 import GUI
from src.project.anim.animations.a1_u3_animation import Aufgabe1

if correct: 
    animation_instance = Aufgabe1(calculator=student_solver)
    app = GUI(default_c=c,
            default_d=d,
            animation_instance=animation_instance)
    display(app.app_layout)
else: 

    print("Your solution is not correct. Please check your implementation.")